# Debug LM Reprojection Optimizer with Synthetic Data

This notebook generates dummy camera poses + 3D landmarks, projects them into 2D with noise/outliers, and feeds frame-by-frame data into `LMGraphReprojOptimizer` (`lm_graph_reproj`).

Key idea: the optimizer stores **2D tracks** immediately, but it only creates 3D landmarks once a track becomes **depth-valid** via the `reg_*` fields.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Project imports
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

import gtsam
from easydict import EasyDict as edict

from point2pose.modules.optimizer.lm_reproj_optimizer import LMGraphReprojOptimizer
from point2pose.data_types.object_frame_data import ObjectFrameData
from point2pose.utils.transform import inverse_SE3

np.set_printoptions(precision=4, suppress=True)


In [ ]:
# -----------------------------
# Plot backend setup (fixes "plots not showing" + enables interactive 3D when possible)
# -----------------------------

# In Jupyter, interactive 3D rotation requires ipympl.
# If you don't have it: `pip install ipympl` and restart the kernel.
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None:
        try:
            import ipympl  # noqa: F401
            ip.run_line_magic('matplotlib', 'widget')
            print('matplotlib backend: widget (ipympl)')
        except Exception:
            ip.run_line_magic('matplotlib', 'inline')
            print('matplotlib backend: inline (install ipympl for interactive 3D)')
except Exception as e:
    print('backend setup skipped:', repr(e))

plt.ion()


In [ ]:
# -----------------------------
# Config
# -----------------------------
rng = np.random.default_rng(42)

OBJ_ID = 0
NUM_FRAMES = 15
NUM_TRACKS = 50

# Camera intrinsics (pixels)
# Wider FOV => more points land in the image (reduces NaNs)
W, H = 640, 480
fx, fy = 300.0, 300.0
cx, cy = W / 2.0, H / 2.0
s = 0.0

K = np.array([
    [fx, s,  cx],
    [0.0, fy, cy],
    [0.0, 0.0, 1.0],
], dtype=float)

# Noise/outliers
PIX_SIGMA = 1.0            # px
DEPTH_SIGMA = 0.01         # meters (noise on camera-frame XYZ)
OUTLIER_PROB = 0.02
MISSING_2D_PROB = 0.0  # keep all visible tracks (reduces NaNs)

# Depth availability (controls when tracks get promoted to 3D landmarks)
DEPTH_AVAILABLE_PROB = 0.8  # promote more landmarks so plots populate

print('K=\n', K)


In [ ]:
# -----------------------------
# Synthetic world + camera trajectory
# -----------------------------

def look_at_rotation_c2w(cam_pos_w, target_w=np.array([0.0, 0.0, 0.0]), up_w=np.array([0.0, 0.0, 1.0])):
    """Return R_c2w for a camera that looks at target (OpenCV-ish: +Z forward, +X right, +Y down)."""
    forward = (target_w - cam_pos_w).astype(float)
    forward /= (np.linalg.norm(forward) + 1e-12)

    right = np.cross(forward, up_w)
    n = np.linalg.norm(right)
    if n < 1e-8:
        right = np.cross(forward, np.array([1.0, 0.0, 0.0]))
        n = np.linalg.norm(right)
    right /= (n + 1e-12)

    down = np.cross(right, forward)
    down /= (np.linalg.norm(down) + 1e-12)

    R_c2w = np.column_stack([right, down, forward])
    # Orthonormalize
    U, _, Vt = np.linalg.svd(R_c2w)
    R_c2w = U @ Vt
    if np.linalg.det(R_c2w) < 0:
        U[:, -1] *= -1
        R_c2w = U @ Vt
    return R_c2w


def make_poses_c2w(num_frames, radius=2.0, height=1.0):
    poses = []
    for i in range(num_frames):
        a = 2.0 * np.pi * i / num_frames
        cam_pos = np.array([radius * np.cos(a), radius * np.sin(a), height], dtype=float)
        R_c2w = look_at_rotation_c2w(cam_pos)
        T = np.eye(4)
        T[:3, :3] = R_c2w
        T[:3, 3] = cam_pos
        poses.append(T)
    return np.stack(poses, axis=0)


def make_landmarks_world(num_tracks, spread_xy=0.75, z_range=(0.2, 1.5)):
    """Landmarks clustered near the look-at target (origin) => fewer behind-camera / off-image points."""
    pts = rng.uniform(-spread_xy, spread_xy, size=(num_tracks, 3)).astype(float)
    pts[:, 2] = rng.uniform(z_range[0], z_range[1], size=(num_tracks,)).astype(float)
    return pts


poses_c2w_gt = make_poses_c2w(NUM_FRAMES)
landmarks_w_gt = make_landmarks_world(NUM_TRACKS)

print('poses_c2w_gt:', poses_c2w_gt.shape)
print('landmarks_w_gt:', landmarks_w_gt.shape)


In [ ]:
# -----------------------------
# Projection + measurement simulation
# -----------------------------

def world_to_cam(T_w2c, Pw):
    Pw_h = np.concatenate([Pw, np.ones((Pw.shape[0], 1))], axis=1)
    Pc_h = (T_w2c @ Pw_h.T).T
    return Pc_h[:, :3]


def project_points(K, Pc):
    """Project camera-frame points Pc (N,3) -> uv (N,2). Returns uv and mask for z>0."""
    z = Pc[:, 2]
    valid = z > 1e-6
    uv = np.full((Pc.shape[0], 2), np.nan, dtype=float)
    x = Pc[valid, 0] / z[valid]
    y = Pc[valid, 1] / z[valid]
    uv[valid, 0] = K[0, 0] * x + K[0, 1] * y + K[0, 2]
    uv[valid, 1] = K[1, 1] * y + K[1, 2]
    return uv, valid


def in_image(uv, W, H, margin=10.0):
    return (
        (uv[:, 0] >= -margin)
        & (uv[:, 0] <= W - 1 + margin)
        & (uv[:, 1] >= -margin)
        & (uv[:, 1] <= H - 1 + margin)
    )


def simulate_frame_measurements(frame_id, T_c2w_gt, landmarks_w):
    """Return dict with visible 2D + optional depth (camera-frame XYZ) for a subset."""
    T_w2c = inverse_SE3(T_c2w_gt)

    Pc = world_to_cam(T_w2c, landmarks_w)  # (N,3)
    uv, front_mask = project_points(K, Pc)
    img_mask = front_mask & in_image(uv, W, H)

    # Randomly drop some 2D observations
    keep = img_mask & (rng.random(NUM_TRACKS) > MISSING_2D_PROB)

    vis_idx = np.where(keep)[0].astype(np.int64)
    vis_uv = uv[keep].copy()

    # Add pixel noise
    vis_uv += rng.normal(0.0, PIX_SIGMA, size=vis_uv.shape)

    # Inject outliers
    outlier = rng.random(vis_uv.shape[0]) < OUTLIER_PROB
    if outlier.any():
        vis_uv[outlier, 0] = rng.uniform(0, W, size=(outlier.sum(),))
        vis_uv[outlier, 1] = rng.uniform(0, H, size=(outlier.sum(),))

    # Per-track measurement uncertainty (what LMGraphReprojOptimizer uses for noise)
    vis_sigma = np.full((vis_uv.shape[0],), PIX_SIGMA, dtype=float)
    vis_sigma[outlier] = PIX_SIGMA * 10.0

    # Depth availability: choose subset of visible tracks to have camera-frame XYZ
    depth_ok = rng.random(vis_uv.shape[0]) < DEPTH_AVAILABLE_PROB
    depth_lids = vis_idx[depth_ok]

    # reg_cur_3d is indexed by global track id (lid)
    reg_cur_3d = np.full((NUM_TRACKS, 3), np.nan, dtype=float)
    for lid in depth_lids:
        reg_cur_3d[lid] = Pc[lid] + rng.normal(0.0, DEPTH_SIGMA, size=(3,))

    # In this synthetic setup, treat depth-available tracks as "valid" and inliers
    reg_valid_idx = depth_lids.astype(np.int64)
    reg_inliers = np.ones((reg_valid_idx.shape[0],), dtype=bool)
    reg_residuals = np.full((reg_valid_idx.shape[0],), PIX_SIGMA, dtype=float)
    reg_uncertainties = np.full((reg_valid_idx.shape[0],), PIX_SIGMA, dtype=float)

    reg_cur_3d_idx = reg_valid_idx.copy()

    return {
        'visible_pts_2d': vis_uv,
        'visible_pts_2d_idx': vis_idx,
        'visible_uncertainties': vis_sigma,
        'reg_cur_3d': reg_cur_3d,
        'reg_cur_3d_idx': reg_cur_3d_idx,
        'reg_valid_idx': reg_valid_idx,
        'reg_inliers': reg_inliers,
        'reg_residuals': reg_residuals,
        'reg_uncertainties': reg_uncertainties,
    }


# quick sanity check on frame 0
m0 = simulate_frame_measurements(0, poses_c2w_gt[0], landmarks_w_gt)
print('frame0 visible:', m0['visible_pts_2d'].shape[0], 'depth tracks:', m0['reg_valid_idx'].shape[0])


In [ ]:
# -----------------------------
# Build ObjectFrameData stream
# -----------------------------

# IMPORTANT: re-seed here so earlier "sanity checks" don't consume RNG and accidentally
# produce a stream with zero depth-promotions.
rng = np.random.default_rng(42)

def make_pose_noisy_w2c(T_c2w_gt, t_sigma=0.01, r_sigma_rad=0.01):
    """Return noisy world_T_cam (w2c)."""
    # noise in camera position (in world) and rotation (in camera frame)
    T = T_c2w_gt.copy()
    T[:3, 3] += rng.normal(0.0, t_sigma, size=(3,))

    # small rotation noise
    w = rng.normal(0.0, r_sigma_rad, size=(3,))
    theta = np.linalg.norm(w)
    if theta > 1e-12:
        axis = w / theta
        Kx = np.array([
            [0, -axis[2], axis[1]],
            [axis[2], 0, -axis[0]],
            [-axis[1], axis[0], 0],
        ], dtype=float)
        Rn = np.eye(3) + np.sin(theta) * Kx + (1 - np.cos(theta)) * (Kx @ Kx)
        T[:3, :3] = Rn @ T[:3, :3]

    return inverse_SE3(T)


intrinsics = K
stream = []
for fid in range(NUM_FRAMES):
    meas = simulate_frame_measurements(fid, poses_c2w_gt[fid], landmarks_w_gt)
    print(f"frame {fid:02d}: visible={meas['visible_pts_2d'].shape[0]:3d} depth_tracks={meas['reg_valid_idx'].shape[0]:3d}")

    data = ObjectFrameData(
        obj_id=OBJ_ID,
        frame_id=fid,
        intrinsics=intrinsics,
        pose=make_pose_noisy_w2c(poses_c2w_gt[fid]),  # IMPORTANT: world_T_cam
        rel_pose=None,
        visible_pts_2d=meas['visible_pts_2d'],
        visible_pts_2d_idx=meas['visible_pts_2d_idx'],
        visible_uncertainties=meas['visible_uncertainties'],
        reg_cur_3d=meas['reg_cur_3d'],
        reg_cur_3d_idx=meas['reg_cur_3d_idx'],
        reg_valid_idx=meas['reg_valid_idx'],
        reg_inliers=meas['reg_inliers'],
        reg_residuals=meas['reg_residuals'],
        reg_uncertainties=meas['reg_uncertainties'],
    )
    stream.append(data)

print('Built stream:', len(stream), 'frames')
print('Example frame:', stream[0])


In [ ]:
# -----------------------------
# Run LM reprojection optimizer + debug outputs
# -----------------------------

cfg = edict({
    'max_iterations': 50,
    'relative_error_tol': 1e-6,
    'absolute_error_tol': 1e-6,
    'lambda_initial': 1e-1,
    'prior_noise_param': [0.05, 0.05, 0.05, 0.1, 0.1, 0.1],
})

opt = LMGraphReprojOptimizer(cfg)

errs_total = []
num_landmarks = []
num_factors = []
results = []

for data in stream:
    E_before = float(opt._graph.error(opt._values)) if hasattr(opt, '_graph') else np.nan
    res = opt.optimize(data)
    E_after = float(opt._graph.error(opt._values)) if hasattr(opt, '_graph') else np.nan

    errs_total.append((data.frame_id, E_before, E_after))
    num_landmarks.append(len(opt.inserted_landmark_ids))
    num_factors.append(int(opt._graph.size()))
    results.append(res)

    print(f"frame {data.frame_id:02d}: E {E_before:.3e} -> {E_after:.3e} | landmarks {num_landmarks[-1]} | factors {num_factors[-1]} | res {res is not None}")

print('done')


In [ ]:
# -----------------------------
# Visualization: optimized poses + 3D landmarks (from opt._values)
# -----------------------------

def extract_poses_landmarks_from_values(values: gtsam.Values, num_frames: int):
    poses_c2w = []
    for fid in range(num_frames):
        X = gtsam.symbol('x', int(fid))
        if values.exists(X):
            try:
                poses_c2w.append(values.atPose3(X).matrix())
            except RuntimeError:
                poses_c2w.append(None)
        else:
            poses_c2w.append(None)

    l_xyz = {}
    for lid in getattr(opt, 'inserted_landmark_ids', []):
        L = gtsam.symbol('l', int(lid))
        if values.exists(L):
            try:
                p = np.asarray(values.atPoint3(L), dtype=float).reshape(3,)
                l_xyz[int(lid)] = p
            except RuntimeError:
                pass

    return poses_c2w, l_xyz


poses_c2w_opt, landmarks_w_opt = extract_poses_landmarks_from_values(opt._values, NUM_FRAMES)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_title('Optimized (poses + landmarks from opt._values)')

# Plot optimized poses
traj = []
for i, T in enumerate(poses_c2w_opt):
    if T is None:
        continue
    p = T[:3, 3]
    traj.append(p)
    ax.scatter(p[0], p[1], p[2], c='tab:green', s=30)

traj = np.array(traj) if len(traj) else np.empty((0, 3))
if len(traj):
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], c='tab:green', alpha=0.6, label='optimized trajectory')

# Plot landmarks
if len(landmarks_w_opt):
    P = np.stack([landmarks_w_opt[k] for k in sorted(landmarks_w_opt.keys())], axis=0)
    ax.scatter(P[:, 0], P[:, 1], P[:, 2], c='tab:purple', s=15, alpha=0.8, label='optimized landmarks')

# Plot GT landmarks for reference
ax.scatter(landmarks_w_gt[:, 0], landmarks_w_gt[:, 1], landmarks_w_gt[:, 2],
           c='tab:red', s=8, alpha=0.25, label='GT landmarks')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Visualization: reprojection "factors" (observed uv vs reprojected uv)
# -----------------------------

def project_world_point(K, T_c2w, Pw):
    """Project a single world point Pw (3,) using pose T_c2w -> uv (2,) (NaNs if behind)."""
    T_w2c = inverse_SE3(T_c2w)
    Pc = (T_w2c @ np.array([Pw[0], Pw[1], Pw[2], 1.0]))[:3]
    if Pc[2] <= 1e-6:
        return np.array([np.nan, np.nan], dtype=float)
    x = Pc[0] / Pc[2]
    y = Pc[1] / Pc[2]
    u = K[0, 0] * x + K[0, 1] * y + K[0, 2]
    v = K[1, 1] * y + K[1, 2]
    return np.array([u, v], dtype=float)


# Choose a frame to visualize
FRAME_TO_PLOT = min(NUM_FRAMES - 1, 5)
T_c2w_est = poses_c2w_opt[FRAME_TO_PLOT]
assert T_c2w_est is not None, 'No optimized pose available for this frame.'

# Observations for this frame (from synthetic stream)
data = stream[FRAME_TO_PLOT]
vis_uv = np.asarray(data.visible_pts_2d, dtype=float)
vis_ids = np.asarray(data.visible_pts_2d_idx, dtype=int)

# Only plot those that have been promoted to landmarks (i.e., exist in values)
obs = []
rep = []
used_ids = []
for uv, lid in zip(vis_uv, vis_ids):
    if int(lid) not in landmarks_w_opt:
        continue
    uv_hat = project_world_point(K, T_c2w_est, landmarks_w_opt[int(lid)])
    if np.isnan(uv_hat).any() or np.isnan(uv).any():
        continue
    obs.append(uv)
    rep.append(uv_hat)
    used_ids.append(int(lid))

obs = np.array(obs) if len(obs) else np.empty((0, 2))
rep = np.array(rep) if len(rep) else np.empty((0, 2))

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.set_title(f'Frame {FRAME_TO_PLOT}: observed vs reprojected (only promoted landmarks)\nN={len(obs)}')

# plot observed + reprojected
if len(obs):
    ax.scatter(obs[:, 0], obs[:, 1], s=30, c='tab:blue', alpha=0.8, label='observed uv')
    ax.scatter(rep[:, 0], rep[:, 1], s=30, c='tab:orange', alpha=0.8, label='reprojected uv')

    # draw residual vectors
    for (u0, v0), (u1, v1) in zip(obs, rep):
        ax.plot([u0, u1], [v0, v1], 'r-', alpha=0.25, linewidth=1)

    err = np.linalg.norm(obs - rep, axis=1)
    ax.text(0.02, 0.98, f'mean |pixel error| = {err.mean():.2f} px\nmedian = {np.median(err):.2f} px',
            transform=ax.transAxes, va='top', ha='left', fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
else:
    ax.text(0.5, 0.5, 'No promoted landmarks visible in this frame yet.', ha='center', va='center')

ax.set_xlim(0, W)
ax.set_ylim(H, 0)  # image coords (v down)
ax.set_xlabel('u (px)')
ax.set_ylabel('v (px)')
ax.grid(True, alpha=0.2)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Quantitative verification of LMGraphReprojOptimizer
# -----------------------------

from math import acos

# 1) Pose error vs ground truth (camera-to-world)
pose_trans_err = []
pose_rot_err = []
valid_pose_frames = []

for fid in range(NUM_FRAMES):
    T_gt = poses_c2w_gt[fid]
    T_est = poses_c2w_opt[fid] if fid < len(poses_c2w_opt) else None
    if T_est is None:
        continue
    valid_pose_frames.append(fid)

    t_gt = T_gt[:3, 3]
    t_est = T_est[:3, 3]
    pose_trans_err.append(np.linalg.norm(t_gt - t_est))

    R_gt = T_gt[:3, :3]
    R_est = T_est[:3, :3]
    R_rel = R_gt.T @ R_est
    tr = np.clip((np.trace(R_rel) - 1.0) / 2.0, -1.0, 1.0)
    pose_rot_err.append(acos(tr))

pose_trans_err = np.array(pose_trans_err)
pose_rot_err = np.array(pose_rot_err)

print('Pose error over frames with estimates:')
print('  frames:', valid_pose_frames)
print('  mean trans err  [m]:', pose_trans_err.mean() if pose_trans_err.size else np.nan)
print('  median trans err[m]:', np.median(pose_trans_err) if pose_trans_err.size else np.nan)
print('  mean rot err    [rad]:', pose_rot_err.mean() if pose_rot_err.size else np.nan)
print('  median rot err  [rad]:', np.median(pose_rot_err) if pose_rot_err.size else np.nan)

# 2) Landmark position error vs ground truth
lm_err = []
for lid, p_est in landmarks_w_opt.items():
    if lid < landmarks_w_gt.shape[0]:
        lm_err.append(np.linalg.norm(p_est - landmarks_w_gt[lid]))

lm_err = np.array(lm_err)
print('\nLandmark error over promoted landmarks:')
print('  num landmarks:', lm_err.size)
print('  mean |p_est - p_gt| [m]:', lm_err.mean() if lm_err.size else np.nan)
print('  median |p_est - p_gt| [m]:', np.median(lm_err) if lm_err.size else np.nan)

# 3) Reprojection error: optimized vs measurements, and GT vs measurements

def project_world_batch(K, T_c2w, Pw_batch):
    T_w2c = inverse_SE3(T_c2w)
    Pw_h = np.concatenate([Pw_batch, np.ones((Pw_batch.shape[0], 1))], axis=1)
    Pc = (T_w2c @ Pw_h.T).T[:, :3]
    z = Pc[:, 2]
    uv = np.full((Pw_batch.shape[0], 2), np.nan, dtype=float)
    valid = z > 1e-6
    x = Pc[valid, 0] / z[valid]
    y = Pc[valid, 1] / z[valid]
    uv[valid, 0] = K[0, 0] * x + K[0, 1] * y + K[0, 2]
    uv[valid, 1] = K[1, 1] * y + K[1, 2]
    return uv, valid

errs_opt = []
errs_gt = []

for fid in range(NUM_FRAMES):
    T_est = poses_c2w_opt[fid] if fid < len(poses_c2w_opt) else None
    if T_est is None:
        continue

    data = stream[fid]
    uv_meas = np.asarray(data.visible_pts_2d, dtype=float)
    lids = np.asarray(data.visible_pts_2d_idx, dtype=int)

    # keep only those with an optimized landmark
    mask = np.array([int(l) in landmarks_w_opt for l in lids], dtype=bool)
    if not mask.any():
        continue

    lids_used = lids[mask]
    uv_used = uv_meas[mask]

    Pw_est = np.stack([landmarks_w_opt[int(l)] for l in lids_used], axis=0)
    Pw_gt = landmarks_w_gt[lids_used]

    uv_hat_opt, valid_opt = project_world_batch(K, T_est, Pw_est)
    uv_hat_gt, valid_gt = project_world_batch(K, poses_c2w_gt[fid], Pw_gt)

    m_opt = valid_opt & ~np.isnan(uv_used).any(axis=1)
    m_gt = valid_gt & ~np.isnan(uv_used).any(axis=1)

    if m_opt.any():
        err = np.linalg.norm(uv_hat_opt[m_opt] - uv_used[m_opt], axis=1)
        errs_opt.extend(err.tolist())
    if m_gt.any():
        err = np.linalg.norm(uv_hat_gt[m_gt] - uv_used[m_gt], axis=1)
        errs_gt.extend(err.tolist())

errs_opt = np.array(errs_opt)
errs_gt = np.array(errs_gt)

print('\nReprojection error vs measured pixels (all frames, all promoted landmarks):')
if errs_opt.size:
    print('  OPT: mean = %.3f px, median = %.3f px' % (errs_opt.mean(), np.median(errs_opt)))
else:
    print('  OPT: no valid reprojection residuals')
if errs_gt.size:
    print('  GT : mean = %.3f px, median = %.3f px' % (errs_gt.mean(), np.median(errs_gt)))
else:
    print('  GT : no valid reprojection residuals')

# 4) Simple pass/fail-style checks (tunable thresholds for this synthetic setup)

if pose_trans_err.size:
    if pose_trans_err.mean() < 0.05 and pose_rot_err.mean() < 0.05 and lm_err.mean() < 0.05:
        print('\n[OK] Pose + landmark errors are small for this synthetic scenario.')
    else:
        print('\n[WARN] Pose/landmark errors are larger than expected. Investigate formulation / noise settings.')

if errs_opt.size and errs_gt.size:
    if errs_opt.mean() <= 2.0 * PIX_SIGMA:
        print('[OK] Optimizer reprojection error is on the order of the injected pixel noise.')
    else:
        print('[WARN] Optimizer reprojection error is significantly larger than the injected pixel noise.')


In [ ]:
# Optional: plot error / graph growth
frames = np.array([f for f, _, _ in errs_total])
E_before = np.array([e0 for _, e0, _ in errs_total], dtype=float)
E_after = np.array([e1 for _, _, e1 in errs_total], dtype=float)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].plot(frames, E_before, 'o--', alpha=0.7, label='before')
axs[0].plot(frames, E_after, 'o-', alpha=0.7, label='after')
axs[0].set_title('Total graph error')
axs[0].set_xlabel('frame')
axs[0].set_yscale('log')
axs[0].grid(True, alpha=0.2)
axs[0].legend()

axs[1].plot(frames, num_landmarks, 'o-')
axs[1].set_title('# landmarks (promoted)')
axs[1].set_xlabel('frame')
axs[1].grid(True, alpha=0.2)

axs[2].plot(frames, num_factors, 'o-')
axs[2].set_title('# factors')
axs[2].set_xlabel('frame')
axs[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Sim(3)-invariant evaluation (Umeyama alignment)
# -----------------------------

from math import acos

def umeyama_sim3(X, Y, with_scale=True):
    """Estimate Sim(3) aligning X->Y. Returns (s, R, t) such that s*R@X + t ~= Y.

    X, Y: (N,3) with N>=3.
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    assert X.shape == Y.shape and X.shape[1] == 3
    n = X.shape[0]
    assert n >= 3

    muX = X.mean(axis=0)
    muY = Y.mean(axis=0)
    Xc = X - muX
    Yc = Y - muY

    Sigma = (Yc.T @ Xc) / n
    U, D, Vt = np.linalg.svd(Sigma)

    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1

    R = U @ S @ Vt

    if with_scale:
        varX = (Xc**2).sum() / n
        s = (np.trace(np.diag(D) @ S)) / max(varX, 1e-12)
    else:
        s = 1.0

    t = muY - s * (R @ muX)
    return float(s), R, t


def apply_sim3_points(P, s, R, t):
    P = np.asarray(P, dtype=float)
    return (s * (R @ P.T)).T + t.reshape(1, 3)


def so3_angle(R_rel):
    tr = np.clip((np.trace(R_rel) - 1.0) / 2.0, -1.0, 1.0)
    return acos(tr)


def get_camera_centers_c2w(poses_c2w):
    centers = []
    fids = []
    for fid, T in enumerate(poses_c2w):
        if T is None:
            continue
        centers.append(T[:3, 3])
        fids.append(fid)
    if not centers:
        return np.empty((0, 3)), []
    return np.stack(centers, axis=0), fids


# Choose correspondences for Sim(3) fit.
# Option A: trajectory camera centers (works even if few landmarks)
C_gt, fids = get_camera_centers_c2w(poses_c2w_gt)
C_est, fids_est = get_camera_centers_c2w(poses_c2w_opt)

# Keep only frames where both exist
common = sorted(set(fids).intersection(fids_est))
Cg = np.stack([poses_c2w_gt[i][:3, 3] for i in common], axis=0)
Ce = np.stack([poses_c2w_opt[i][:3, 3] for i in common], axis=0)

if len(common) >= 3:
    s_sim3, R_sim3, t_sim3 = umeyama_sim3(Ce, Cg, with_scale=True)
    print(f"Sim(3) from trajectory: scale={s_sim3:.4f}")
else:
    # Fallback: use landmarks if we have them
    ids = sorted(set(landmarks_w_opt.keys()).intersection(range(landmarks_w_gt.shape[0])))
    if len(ids) >= 3:
        Xe = np.stack([landmarks_w_opt[i] for i in ids], axis=0)
        Yg = np.stack([landmarks_w_gt[i] for i in ids], axis=0)
        s_sim3, R_sim3, t_sim3 = umeyama_sim3(Xe, Yg, with_scale=True)
        print(f"Sim(3) from landmarks: scale={s_sim3:.4f}")
    else:
        raise RuntimeError('Need at least 3 common camera centers or 3 common landmarks for Sim(3) alignment.')


# Evaluate pose errors after Sim(3) alignment
trans_err_aligned = []
rot_err_aligned = []

for fid in common:
    T_gt = poses_c2w_gt[fid]
    T_est = poses_c2w_opt[fid]

    # Align estimated pose by left-multiplying similarity on world frame
    # Rotation: R' = R_sim3 * R_est
    # Translation (camera center): c' = s*R*c + t
    R_gt = T_gt[:3, :3]
    c_gt = T_gt[:3, 3]

    R_est = T_est[:3, :3]
    c_est = T_est[:3, 3]

    R_est_aligned = R_sim3 @ R_est
    c_est_aligned = (s_sim3 * (R_sim3 @ c_est)) + t_sim3

    trans_err_aligned.append(np.linalg.norm(c_gt - c_est_aligned))
    rot_err_aligned.append(so3_angle(R_gt.T @ R_est_aligned))

trans_err_aligned = np.array(trans_err_aligned)
rot_err_aligned = np.array(rot_err_aligned)

print('\nPose error AFTER Sim(3) alignment (trajectory-based):')
print('  mean trans err [m]:', float(trans_err_aligned.mean()) if trans_err_aligned.size else np.nan)
print('  median trans err [m]:', float(np.median(trans_err_aligned)) if trans_err_aligned.size else np.nan)
print('  mean rot err [rad]:', float(rot_err_aligned.mean()) if rot_err_aligned.size else np.nan)
print('  median rot err [rad]:', float(np.median(rot_err_aligned)) if rot_err_aligned.size else np.nan)


# Evaluate landmark errors after Sim(3) alignment (only for promoted landmarks)
ids = sorted(set(landmarks_w_opt.keys()).intersection(range(landmarks_w_gt.shape[0])))
if len(ids):
    P_est = np.stack([landmarks_w_opt[i] for i in ids], axis=0)
    P_gt = np.stack([landmarks_w_gt[i] for i in ids], axis=0)
    P_est_aligned = apply_sim3_points(P_est, s_sim3, R_sim3, t_sim3)
    lm_err_aligned = np.linalg.norm(P_gt - P_est_aligned, axis=1)
    print('\nLandmark error AFTER Sim(3) alignment:')
    print('  num landmarks:', len(ids))
    print('  mean |p_gt - Sim3(p_est)| [m]:', float(lm_err_aligned.mean()))
    print('  median |p_gt - Sim3(p_est)| [m]:', float(np.median(lm_err_aligned)))
else:
    print('\n(No promoted landmarks to evaluate.)')


# Quick visualization: GT vs aligned estimated trajectory
Cg_aligned = apply_sim3_points(Ce, s_sim3, R_sim3, t_sim3)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title('Trajectory: GT vs Sim(3)-aligned estimate')
ax.plot(Cg[:, 0], Cg[:, 1], Cg[:, 2], 'b-o', markersize=3, alpha=0.8, label='GT')
ax.plot(Cg_aligned[:, 0], Cg_aligned[:, 1], Cg_aligned[:, 2], 'g-^', markersize=3, alpha=0.8, label='Sim3-aligned est')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
